In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

from sklearn.utils.class_weight import compute_class_weight

In [3]:
train_df = pd.read_csv("../dataset/train_split.csv")
val_df = pd.read_csv("../dataset/val_split.csv")

class_names = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]

print("Jumlah training   :", len(train_df))
print("Jumlah validation :", len(val_df))

Jumlah training   : 21947
Jumlah validation : 5487


In [4]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(range(len(class_names))),
    y=train_df["label_index"]
)

class_weight_dict = dict(
    enumerate(class_weights)
)

print("Class weights:")

for index, weight in class_weight_dict.items():
    print(f"{class_names[index]:10} : {weight:.4f}")

Class weights:
angry      : 1.0226
disgust    : 10.3475
fear       : 1.0088
happy      : 0.5532
neutral    : 0.8052
sad        : 0.8310
surprise   : 1.4685


In [5]:
def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=1)

    image = tf.cast(image, tf.float32)
    image = image / 255.0

    return image, label

In [6]:
BATCH_SIZE = 32

train_weights = train_df["label_index"].map(
    class_weight_dict
).astype("float32").values


def load_image_with_weight(image_path, label, sample_weight):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=1)
    image = tf.cast(image, tf.float32)
    image = image / 255.0

    return image, label, sample_weight


def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=1)
    image = tf.cast(image, tf.float32)
    image = image / 255.0

    return image, label


train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["image_path"].values,
        train_df["label_index"].values,
        train_weights
    )
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].values,
        val_df["label_index"].values
    )
)

train_dataset = train_dataset.map(
    load_image_with_weight,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_dataset = val_dataset.map(
    load_image,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_dataset = train_dataset.shuffle(
    len(train_df),
    seed=42
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = val_dataset.batch(
    BATCH_SIZE
).prefetch(tf.data.AUTOTUNE)

In [7]:
data_augmentation = Sequential([
    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.05),

    tf.keras.layers.RandomZoom(0.10),

    tf.keras.layers.RandomTranslation(
        height_factor=0.05,
        width_factor=0.05
    )
])

In [8]:
from tensorflow.keras import Input

model = Sequential([
    Input(shape=(48, 48, 1)),

    data_augmentation,

    Conv2D(
        32,
        (3, 3),
        activation="relu"
    ),
    MaxPooling2D((2, 2)),

    Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),
    MaxPooling2D((2, 2)),

    Flatten(),

    Dense(
        128,
        activation="relu"
    ),
    Dropout(0.5),

    Dense(
        7,
        activation="softmax"
    )
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 6400)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       819,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 839,047 (3.20 MB)

 Trainable params: 839,047 (3.20 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
images, labels, weights = next(iter(train_dataset))

print("Shape images :", images.shape)
print("Shape labels :", labels.shape)
print("Shape weights:", weights.shape)

print("Contoh labels :", labels[:10].numpy())
print("Contoh weights:", weights[:10].numpy())

Shape images : (32, 48, 48, 1)
Shape labels : (32,)
Shape weights: (32,)
Contoh labels : [5 2 6 0 2 2 4 5 4 5]
Contoh weights: [0.8309795  1.0087792  1.4685179  1.022598   1.0087792  1.0087792
 0.80515814 0.8309795  0.80515814 0.8309795 ]


In [11]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20
)

Epoch 1/20


d:\JamTerbang\DL\KlasifikasiEkspresi\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


686/686 ━━━━━━━━━━━━━━━━━━━━ 21s 24ms/step - accuracy: 0.2043 - loss: 1.9238 - val_accuracy: 0.2285 - val_loss: 1.8545
Epoch 2/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.2265 - loss: 1.8601 - val_accuracy: 0.2289 - val_loss: 1.8433
Epoch 3/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.2450 - loss: 1.8289 - val_accuracy: 0.2936 - val_loss: 1.8017
Epoch 4/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.2719 - loss: 1.7963 - val_accuracy: 0.2522 - val_loss: 1.8265
Epoch 5/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.2900 - loss: 1.7818 - val_accuracy: 0.2779 - val_loss: 1.7803
Epoch 6/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.3071 - loss: 1.7484 - val_accuracy: 0.3375 - val_loss: 1.7031
Epoch 7/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.3126 - loss: 1.7283 - val_accuracy: 0.3200 - val_loss: 1.7348
Epoch 8/20
686/686 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.3302 - loss: 1.7022 - val_accurac

In [14]:
y_true_combo = []
y_pred_combo = []

for images, labels in val_dataset:
    predictions = model.predict(images, verbose=0)
    predicted_labels = np.argmax(predictions, axis=1)

    y_true_combo.extend(labels.numpy())
    y_pred_combo.extend(predicted_labels)

y_true_combo = np.array(y_true_combo)
y_pred_combo = np.array(y_pred_combo)

print("Jumlah data validation:", len(y_true_combo))

Jumlah data validation: 5487


In [15]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true_combo,
        y_pred_combo,
        target_names=class_names,
        zero_division=0
    )
)

              precision    recall  f1-score   support

       angry       0.33      0.21      0.26       767
     disgust       0.07      0.66      0.12        76
        fear       0.32      0.17      0.22       777
       happy       0.71      0.60      0.65      1417
     neutral       0.44      0.40      0.42       973
         sad       0.31      0.31      0.31       943
    surprise       0.46      0.68      0.55       534

    accuracy                           0.41      5487
   macro avg       0.37      0.43      0.36      5487
weighted avg       0.45      0.41      0.42      5487



## Kesimpulan Experiment Class Weight + Data Augmentation

Kombinasi class weight dan data augmentation belum memberikan peningkatan performa pada CNN baseline. Validation accuracy terbaik yang diperoleh adalah sekitar 42.04%, lebih rendah dibandingkan baseline sebesar 51.08%.

Kombinasi ini meningkatkan recall kelas Disgust hingga 0.66, tetapi precision Disgust hanya 0.07 dan F1-score tetap 0.12. Hal ini menunjukkan bahwa model terlalu sering memprediksi kelas Disgust sehingga menghasilkan banyak false positive.

Macro F1 berada di sekitar 0.43, tetapi weighted F1 dan accuracy mengalami penurunan. Oleh karena itu, kombinasi class weight dan data augmentation tidak digunakan sebagai model utama.